### From protocol to pathway activity

In this tutorial, we will be diving into the protocols you have extracted. We will follow three main steps:
1. Integrate the concentration and duration of each growth factor into a compound-level score
2. Derive pathway scores based on compound action
3. Fit a multivariate logistic regression model to estimate the contribution of each pathway to the final cell type composition.

For the last step, we will be using the cell type quantification you derived from the scRNAseq data to find the connection between pathway activation and cell type differentiation.

To be efficient, we will do this over all protocols at once. Save all your extracted jsons in a folder and we will work with them all at once.

Let's first import our packages. As you have probably seen, the protocol information is saved in a file format called json. In this tutorial, you will learn how to parse it an extract the correct information.

In [ ]:
import pandas as pd
import json
import os
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

Based on the differentiation stage of interest, you have to decide which steps of the protocol to include in the analysis. For instance, if you wish to compare the pathway contributions that lead to foregut, hindgut and midgut, you will have to only include the steps (compounds and concentrations) that are used in differentiating cells until these stages. 

Let's begin by reading a json file containing one of your extractions into a variable called a dictionary. Later, we will process the entire folder.

In [ ]:
path_to_protocol = 'data/protocols/fileName.json' #replace with the name of a file in the protocols folder

with open(path_to_protocol) as json_data:
    prot = json.load(json_data)
    
prot

As you can see, there are a lot of different fields in the json. They contain the same information you have extracted in the online form:
- ```prot['cellLines']``` contains information about each starting cell line, such as ```type```, which is iPSC or ESC, and ```name```.
- ```prot['targetCell']``` contains the differentiation target you specified in the form
- ```prot['publicationId']``` contains the PMID of the publication

The actual protocol information is split per step in the ```steps``` field. For instance, to access the first step of the differentiation, we can do ```prot['steps'][1]```. Pay attention that the step numerotation follows the same logic as the online form, where step 0 is the stem cell culturing protocol, not the differentiation.

In [ ]:
prot['steps'][1]

Let's see how many steps we have in the protocol.

In [ ]:
print ('There are', len(prot['steps']), 'including step 0')

You can also see that each step has different fields for basal media composition, serums and supplements, and growth factors. Let's print the list of growth factors in step 1.

In [ ]:
for i in range(len(prot['steps'][1]['growthFactors'])):
    print (prot['steps'][1]['growthFactors'][i]['name'])

Feel free to explore more of the entries and compare them with the online form. 

Once you are familiar with the structure, we can move on to the next step, which is extracting the relevant information. In the next cell I am defining a few functions to help us do that:

- ```extract_growth_factors_into_df``` will parse the json until a chosen step and extract per step the duration, and the growth factor names and corresponding concentrations.
- ```compute_growth_factor_exposure``` groups the growth factors by name and publication and calculates the exposure of each compound, defined as the summed concentration x time per step.
- ```z_score_exposure```  Z-score scales the compound exposures among all publications to make them comparable.

In [ ]:

def extract_growth_factors_into_df(protocol, untilStep, protocolID):
    """
    Extract step, duration (in days), growth factor name and concentration
    into a pandas DataFrame.

    Columns:
    - step
    - duration (in days, float)
    - growth_factor
    - concentration
    """
    rows = []

    for step in protocol.get("steps", []):
        step_number = step.get("stepNumber")

        if step_number is None or step_number == 0 or step_number > untilStep:
            continue

        # ---- Normalize duration to days ----
        duration_list = step.get("duration", [])
        duration_days = None

        if duration_list:
            durations = []
            for d in duration_list:
                value = d.get("durationHours")
                dtype = d.get("durationType")

                if value is None:
                    continue

                try:
                    value = float(value)
                except:
                    continue

                if dtype == "hours":
                    durations.append(value / 24)
                elif dtype == "days":
                    durations.append(value)
                else:
                    durations.append(value)  # fallback

            if durations:
                duration_days = sum(durations)  # total duration per step

        # ---- Extract growth factors ----
        growth_factors = step.get("growthFactors", [])

        if growth_factors:
            for gf in growth_factors:
                rows.append({
                    "step": step_number,
                    "duration": duration_days,
                    "growth_factor": gf.get("name"),
                    "concentration": gf.get("concentration")
                })
        else:
            rows.append({
                "step": step_number,
                "duration": duration_days,
                "growth_factor": None,
                "concentration": None
            })

    df = pd.DataFrame(rows)

    #Add PMID 
    df['protocol'] = protocolID

    return df

import pandas as pd
import numpy as np

def compute_growth_factor_exposure(df):
    """
    Compute exposure per (protocol, growth_factor) and z-score
    across protocols for each growth factor.

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain:
        - 'protocol'
        - 'duration'
        - 'growth_factor'
        - 'concentration'

    Returns
    -------
    pandas.DataFrame
        Columns:
        - protocol
        - growth_factor
        - exposure
        - z_exposure (scaled per growth factor across protocols)
    """

   
    # Ensure numeric
    df["concentration"] = pd.to_numeric(df["concentration"], errors="coerce")

    # Drop invalid rows
    df = df.dropna(subset=["protocol", "growth_factor", "duration", "concentration"])

    # Compute exposure
    df["exposure"] = df["duration"] * df["concentration"]

    # Aggregate per protocol + growth factor
    df_grouped = (
        df.groupby(["protocol", "growth_factor"], as_index=False)["exposure"]
        .sum()
    )

    return df_grouped


def zscore(x):
    """ 
    Helper function to compute z score per input array
    """
   
    std = x.std()
    if std == 0 or np.isnan(std):
        return pd.Series([0] * len(x), index=x.index)
    return (x - x.mean()) / std


def z_score_exposure (df):
    
    # Z-score per growth factor across protocols
    def zscore(x):
        std = x.std()
        if std == 0 or np.isnan(std):
            return pd.Series([0] * len(x), index=x.index)
        return (x - x.mean()) / std

    df["z_exposure"] = (
        df.groupby("growth_factor")["exposure"]
        .transform(zscore)
    )

    return df

Let's first extract the growth factor information.

In [ ]:
#if you have multiple protocols from the same publication, it is better to use something other than PMID as protocolID. 

growth_factor_df = extract_growth_factors_into_df(protocol=prot, untilStep=3, protocolID='27748754') #if you have multiple protocols, it is better to use a differnt ID to 
growth_factor_df.head()

Now let's calculate the scaled exposure values.

In [ ]:
exposure_matrix = compute_growth_factor_exposure(growth_factor_df)
exposure_matrix.head()

## All protocols

To make the exposures comparable across different growth factors, we must scale them. We use Z-score scaling to convert the exposure values to mean 0 and unit variance for each compound acrosss all protocols. This process helps in reshaping the data as deviation from the mean. However, if we apply Z scaling to only one protocol, the value will essentially be zero. Therefore, it is now time to combine the data from all your protocols before we proceed.

However, before we can do that, we must decide the number of steps we take into consideration in our analysis, per protocol. Fill in the dictionary below with the correct step number for each protocol json file name.

In [ ]:
dict_steps = {

    'PMID1_participantID.json' : stepNo,  #replace with your own file names
    'PMID2_participantID.json' : stepNo, 
   
}

As stated before, make sure you have all your protocols within one folder and adjust the path below accordingly. We will be looping through all the files in that folder, so make sure there are no other files other than the jsons in there. In essence, we read every protocol in the folder, process it and we concatenate the resulting exposure matrices at the end. 

In [ ]:
path_to_protocols_folder = 'data/protocols' #adjust this with your actual path

#empty dictionary to store the resulting dataframes before concatenation
all_dfs = []

#Loop through all files in the specified folder
for path_to_protocol in os.listdir(path_to_protocols_folder):
    #Load the  json
    with open(os.path.join(path_to_protocols_folder, path_to_protocol)) as json_data:
        prot = json.load(json_data)
    
    #Get the final step from the dictionary defined above
    untilStep = dict_steps[path_to_protocol]

    #Extract growth factors and calculate exposure
    
    growth_factor_df = extract_growth_factors_into_df(prot, untilStep, path_to_protocol)
    exposure_df = compute_growth_factor_exposure(growth_factor_df)

     # Store
    all_dfs.append(exposure_df)

# Concatenate everything
final_df = pd.concat(all_dfs, ignore_index=True)

final_df.head()
    

Now we can scale all the growth factors.

In [ ]:
scaled_exposure_matrix = z_score_exposure (final_df)
scaled_exposure_matrix

## Pathway Clustering

The next step is now to cluster your growth factors into pathways. The way we do this is simple, we define a pathway as the sum of scaled growth factor exposure values that activate the pathway and subtract the sum of factors that inhibit the pathway. 

This requires you to define the pathways after consulting literature. Use the template below to code it. Let's start by taking a look at all the growth factors we have information about. Also take this opportunity to make sure the same compound is not present under different names in your pipeline (Ex. retinoic acid and RA). If so, correct it in the online form, download the updated json and run the code above again to process and concatenate all protocols.

In [ ]:
#Print all growth factors
print (scaled_exposure_matrix.growth_factor.unique())

In [ ]:
# Add here name normalization 
name_map = {
    "ACTIVIN A": "Activin A", #so our ACTIVIN A gets re-named to Activin A to match the other entries
    "Sonic Hedgehog": "SHH"
}


In [ ]:
#Sort into pathways
PATHWAYS = {
"BMP_signal": {
    "activators": ["BMP4"],
    "inhibitors": ["Dorsomorphin"],
},
"TGFb_Activin_signal": {
        "activators": ["Activin A"],
        "inhibitors": ["SB431542"],
    },
}

In [ ]:
#Combine all pathway terms and proticol exposure values in one matrix
pathway_matrix = scaled_exposure_matrix.copy()
pathway_matrix ["growth_factor"] = pathway_matrix ["growth_factor"].replace(name_map)

# convert into protocol x growth factor matrix
wide = (
    pathway_matrix.pivot_table(
        index="protocol",
        columns="growth_factor",
        values="z_exposure",
        aggfunc="sum",
        fill_value=0
    )
)
wide.head()

In [ ]:
## Calculate pathway scores

# add missing columns as 0 so code does not fail
all_factors = set()
for pdef in PATHWAYS.values():
    all_factors.update(pdef["activators"])
    all_factors.update(pdef["inhibitors"])

for factor in all_factors:
    if factor not in wide.columns:
        wide[factor] = 0

# compute pathway scores
for pathway_name, pdef in PATHWAYS.items():
    activator_score = wide[pdef["activators"]].sum(axis=1)
    inhibitor_score = wide[pdef["inhibitors"]].sum(axis=1)
    wide[pathway_name] = activator_score - inhibitor_score

# keep only pathway columns if you want
pathway_scores = wide[list(PATHWAYS.keys())].reset_index()

pathway_scores.head()

Now that we have our pathway exposure scores for each protocol, the goal is to fit a multivariate linear regression model that will tell us the contribution of each pathway to the final cell type. This is where we bring in the cellular composition percentages calculated previously.

In [ ]:
#Read in the cell composition for each protocol

# list of files
files = [
    "data/3802619/cell_type_quantification.csv",
    #"data/.../cell_type_quantification.csv",
    #"data/.../cell_type_quantification.csv",
    # add more here
]

dfs = []

for f in files:
    df = pd.read_csv(f)

    # extract protocol ID (folder name)
    protocol_id = os.path.basename(os.path.dirname(f))

    df["protocol"] = protocol_id
    dfs.append(df)

# concatenate all
cell_comp = pd.concat(dfs, ignore_index=True)

cell_comp.head()

In [ ]:
Y_long = cell_comp.copy()

#Convert to protocol x cell type table, where values are percentages
Y_wide = (
    Y_long.pivot_table(
        index="protocol",
        columns="cell_type",
        values="percentage_cells",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

Y_wide.head()

This is where you can subset to cell types of interest.

In [ ]:
print (Y_wide.columns)

cell_types_to_keep = ['protocol', 'Early epithelial endoderm', 'Foregut / gut tube endoderm'] #as an example
Y_wide = Y_wide[cell_types_to_keep]

print (Y_wide)

In [ ]:
#  Make sure protocol is only the pmid
X_df = pathway_scores.copy()
X_df['protocol'] = X_df['protocol'].str.split('_').str[0]

In [ ]:

#Merge the pathway scores with the percentage scores 
df_model = pd.merge(X_df, Y_wide, on="protocol", how="inner")

#Define which are the model features (pathway scores) and the regression targets (cell composition %)
feature_cols = [c for c in pathway_scores.columns if c != "protocol"]
lineage_cols = [c for c in Y_wide.columns if c != "protocol"]


#Scale pathway scores (you can do a test without this step)
#scaler = StandardScaler()
#X_scaled = scaler.fit_transform(df_model[feature_cols])

X_scaled = X_df

#Fit a linear regression model for each 
models = {}
coef_dict = {}
predictions = {}

for lineage in lineage_cols:
    y = df_model[lineage]

    model = LinearRegression(fit_intercept=True)
    model.fit(X_scaled, y)

    y_pred = model.predict(X_scaled)

    models[lineage] = model
    predictions[lineage] = y_pred
    coef_dict[lineage] = model.coef_

    df_model[lineage + "_fit"] = y_pred

    eq_terms = [f"{model.intercept_:.3f}"]
    for f, c in zip(feature_cols, model.coef_):
        eq_terms.append(f"{c:.3f}·{f}")

    print(f"\n=== {lineage} ===")
    print("Lineage =", " + ".join(eq_terms))
    print(f"R² = {r2_score(y, y_pred):.3f}, MAE = {mean_absolute_error(y, y_pred):.3f}")


#Extract and print the matrix of coefficients from the regression 
coef_df = pd.DataFrame(coef_dict, index=feature_cols).T
print("\nCoefficient matrix:")
print(coef_df)

Now let's visualise our model results. We will look for two things: predicted vs true correlation and pathway coefficients. 

In [ ]:
outdir = "pathway_regression_results"
os.makedirs(outdir, exist_ok=True)

for lineage in lineage_cols:
    y_true = df_model[lineage]
    y_pred = df_model[lineage + "_fit"]

    plt.figure(figsize=(4, 4))
    plt.scatter(y_true, y_pred, s=70)

    min_v = min(y_true.min(), y_pred.min())
    max_v = max(y_true.max(), y_pred.max())
    plt.plot([min_v, max_v], [min_v, max_v], "k--") # add diagonal line for full correlation

    for i, pid in enumerate(df_model["protocol"]):
        plt.text(y_true.iloc[i] + 0.3, y_pred.iloc[i], str(pid), fontsize=8) #write the equation

    plt.text(
        0.05, 0.95,
        f"R² = {r2:.2f}",
        transform=plt.gca().transAxes,
        ha="left",
        va="top",
        fontsize=10
    ) #write the r2

    plt.xlabel("True (%)")
    plt.ylabel("Predicted (%)")
    plt.title(lineage)
    plt.tight_layout()
    plt.savefig(
        os.path.join(outdir, f"{lineage.replace(' ', '_').replace('/', '_')}_fit.svg"),
        dpi=300
    )
    plt.show()

In [ ]:
plt.figure(figsize=(1.2 * len(feature_cols) + 3, 0.45 * len(lineage_cols) + 3))
im = plt.imshow(coef_df.values, aspect="auto", interpolation="nearest")
plt.colorbar(im, label="Linear weight")

plt.xticks(range(len(feature_cols)), feature_cols, rotation=45, ha="right")
plt.yticks(range(len(lineage_cols)), lineage_cols)

for i in range(coef_df.shape[0]):
    for j in range(coef_df.shape[1]):
        plt.text(j, i, f"{coef_df.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.xlabel("Signaling pathway")
plt.ylabel("Cell type")
plt.title("Linear weights")
plt.tight_layout()
plt.savefig(os.path.join(outdir, "lineage_signal_weights.svg"), dpi=300)
plt.show()